# Exploring Jev

Initial experimentation with Jev (`typesafe-ai/jev`). Using Vercel AI Gateway, called over plain HTTP. 

1. Setup and a raw response
2. The three kinds of question Jev can answer (yes/no, choice, score)
3. How confidence behaves on clear and unclear input
4. Parallelising several questions in one call, vs. separate calls
5. Where it does not work well

Every request and response is appended to `results/phase1_raw.jsonl`. The file is rewritten on each
run, and responses are never cached.

**Naming:** Jev calls the yes/no question type `Noul`, although Vercel's API `/v1/evaluate` calls it `boolean` and its answer field is `probability`. It is the same thing under a different name. Choice and score answers also include a `confidence` value.

## 1. Setup

In [1]:
import json
import statistics
import time
from datetime import date
from pathlib import Path

import pandas as pd

from jev_compare import jev

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 70)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_LOG = ROOT / "results" / "phase1_raw.jsonl"
RAW_LOG.parent.mkdir(exist_ok=True)
RAW_LOG.write_text("")  # start fresh each run


FAILURES = []  # every non-2xx attempt, kept so the run can report them
MAX_ATTEMPTS = 6


def call(label, state, questions):
    """One Jev call, retrying transient failures (429/5xx) with backoff.

    Every attempt, failed or not, is appended to the raw log. Reported latency is that of the
    successful attempt only (retries and backoff sleeps are excluded).
    """
    for attempt in range(1, MAX_ATTEMPTS + 1):
        raw = jev.evaluate_raw(state, questions)
        with RAW_LOG.open("a") as f:
            f.write(
                json.dumps(
                    {
                        "label": label,
                        "attempt": attempt,
                        "status": raw.status_code,
                        "latency_s": raw.latency_s,
                        "request": raw.request,
                        "body": raw.body,
                    }
                )
                + "\n"
            )
        if raw.ok:
            return jev.parse(raw)
        FAILURES.append((label, attempt, raw.status_code))
        if raw.status_code not in (429, 500, 502, 503, 504) or attempt == MAX_ATTEMPTS:
            return jev.parse(raw)  # raises JevError
        time.sleep(2**attempt)


def show(res, label=None):
    """Answers, usage, latency and list-price cost (gateway routing block omitted)."""
    print(
        f"{label or ''}  latency={res.latency_s:.2f}s  "
        f"tokens in/out={res.input_tokens}/{res.output_tokens}  market cost=${res.market_cost:.7f}"
    )
    print(json.dumps(res.answers, indent=2))

### Connectivity check and the full raw response

The key is read from `.env` (`AI_GATEWAY_API_KEY`). The next cell prints the model listing, then one complete response, unedited, including the gateway routing details. Later sections leave those details out, but they are kept in the raw log.

In [2]:
print("model listing:", json.dumps(jev.list_models(), indent=2))
print("run date:", date.today().isoformat())

ticket = (
    "Hi, I was charged twice for my subscription this month and I'd like the extra payment "
    "back please. Order #48213."
)
r0 = call(
    "setup-connectivity",
    ticket,
    {"refund": jev.boolean("Is the customer asking for money back?")},
)
print(f"\nHTTP latency: {r0.latency_s:.2f}s\n")
print(json.dumps(r0.raw.body, indent=2))

model listing: {
  "models": [
    {
      "name": "jev",
      "description": "Jev is TypeSafe AI\u2019s System One evaluation model for fast, structured decisions in software. It evaluates shared state against typed questions and returns choices, scores, and boolean probabilities, supporting classification, routing, rubric-based assessment, and automated verification. Multiple questions can be evaluated in parallel within a single request.",
      "release_date": "2026-09-15"
    }
  ]
}
run date: 2026-09-20



HTTP latency: 0.58s

{
  "model": "typesafe-ai/jev",
  "answers": {
    "refund": {
      "type": "boolean",
      "probability": 0.99
    }
  },
  "usage": {
    "inputTokens": 303,
    "outputTokens": 20
  },
  "providerMetadata": {
    "typesafe": {
      "confidence": {}
    },
    "gateway": {
      "routing": {
        "originalModelId": "typesafe-ai/jev",
        "resolvedProvider": "typesafe-ai",
        "fallbacksAvailable": [],
        "planningReasoning": "System credentials planned for: typesafe-ai. Total execution order: typesafe-ai(system)",
        "canonicalSlug": "typesafe-ai/jev",
        "finalProvider": "typesafe-ai",
        "modelAttemptCount": 1,
        "modelAttempts": [
          {
            "canonicalSlug": "typesafe-ai/jev",
            "success": true,
            "providerAttemptCount": 1,
            "providerAttempts": [
              {
                "provider": "typesafe-ai",
                "credentialType": "system",
                "success": tr

**Model version:** The response gives only `"model": "typesafe-ai/jev"`, and there is no version string
(such as `jev-1.13.0`) in the response body, the response headers or the models listing, which gives
only a release date (2026-09-15). For this project, "the Jev that was tested" therefore means
`typesafe-ai/jev` as served on the run date above.

Note also the difference between `gateway.cost` and `gateway.marketCost`. In the introductory period, Vercel gateway charged `0` on this account, while `marketCost` holds the list price, so this notebook uses `marketCost` for cost comparisons.

## 2. Jev has three modes of operation

Unlike a regular LLM, Jev does not generate. Instead, we can ask it questions with specific output types: confidence in yes/no, multiple choice, or use a scoring range.

In [3]:
STATE = (
    "Subject: Export failing + charged twice\n\n"
    "Hi team, since yesterday's update the 'Export report' button on the Analytics page throws "
    "an error and I can't download my monthly report, which my finance lead needs by Friday. "
    "Separately, I can see two identical $49 charges on my card for this month. Could you "
    "reverse the duplicate one? Order #48213. Thanks, Dana"
)

DEPARTMENTS = {
    "billing": "payments, invoices, duplicate or incorrect charges, refunds",
    "technical": "bugs, errors, crashes, or features not working as designed",
    "account": "login, password, permissions, or profile changes",
    "sales": "pricing questions, plan upgrades, new purchases, quotes",
    "other": "anything that fits none of the above",
}
SEVERITY = [
    "low: cosmetic or minor; a workaround exists and nobody is blocked",
    "medium: a feature is broken for some users, with a workaround or a deadline days away",
    "high: core functionality is down for many users, or data loss/security risk, no workaround",
]

r_yes_no = call(
    "s2-boolean",
    STATE,
    {"refund": jev.boolean("Does this message ask for a refund or reversal of a charge?")},
)
r_choice = call(
    "s2-choice",
    STATE,
    {"route": jev.choice("Which team should handle this ticket first?", DEPARTMENTS)},
)
r_score = call(
    "s2-score", STATE, {"severity": jev.score("Rate the severity of the reported bug.", SEVERITY)}
)

In [4]:
show(r_yes_no, "BOOLEAN")

BOOLEAN  latency=0.35s  tokens in/out=362/20  market cost=$0.0000152
{
  "refund": {
    "type": "boolean",
    "probability": 0.99
  }
}


In [5]:
show(r_choice, "CHOICE")

CHOICE  latency=0.38s  tokens in/out=470/52  market cost=$0.0000197
{
  "route": {
    "type": "choice",
    "choice": "billing",
    "probabilities": {
      "account": 0,
      "sales": 0,
      "billing": 0.56,
      "other": 0,
      "technical": 0.44
    },
    "confidence": 0.44
  }
}


In [6]:
show(r_score, "SCORE")

SCORE  latency=0.39s  tokens in/out=427/17  market cost=$0.0000179
{
  "severity": {
    "type": "score",
    "score": 1.01,
    "probabilities": {
      "0": 0,
      "1": 0.99,
      "2": 0.01
    },
    "confidence": 0.98
  }
}


**Reading the output.** The yes/no answer is a single `probability`. The choice answer gives the
selected option, a probability for each option, and a `confidence`. The score answer gives a `score`
on a scale starting at 0 (0 = low, 1 = medium, 2 = high), a probability for each level, and a
`confidence`.

The choice answer is worth a closer look. The ticket reports two problems, an export bug and a
duplicate charge, and the question asks which team should take it first. Jev splits close to evenly
between `billing` and `technical` and reports low confidence, while the yes/no and score questions
about the same text are answered confidently. The ticket is unclear for routing purposes, and the
answer reflects that.

## 3. How confidence behaves on clear and unclear tickets

Each question type is tried on clear tickets and on unclear ones, with the same question each time.
The table shows the answer and how sure Jev is. For yes/no questions the probability itself is the
measure (how far it is from 0.5). For choice and score questions the table shows the highest option
probability and the reported `confidence`.

In [7]:
Q_REFUND = jev.boolean("Is the customer asking for a refund?")
Q_ROUTE = jev.choice("Which team should handle this ticket?", DEPARTMENTS)
Q_SEV = jev.score("Rate the severity of the reported bug.", SEVERITY)

cases = [
    # (question type, clarity, text)
    ("boolean", "clear", "I was charged twice, please refund the duplicate $49 payment."),
    (
        "boolean",
        "clear",
        "Love the new dashboard! Quick question: how do I change the chart colours?",
    ),
    ("boolean", "clear", "Cancel my plan and give me my money back, this product never worked."),
    (
        "boolean",
        "unclear",
        "Hi, there's a $49 charge on my card from you. I'm not sure it's the right amount for my plan, could someone take a look? I might just cancel depending on what you find.",
    ),
    (
        "boolean",
        "unclear",
        "The invoice looks higher than I expected. I'm not asking for anything yet, just flagging it, but I'm not happy either.",
    ),
    (
        "boolean",
        "unclear",
        "If the export bug isn't fixed by Friday I'll have to reconsider whether this is worth paying for.",
    ),
    ("choice", "clear", "I was charged twice for order #48213, please reverse one."),
    ("choice", "clear", "The Analytics page throws a 500 error every time I open it."),
    ("choice", "clear", "I forgot my password and the reset email never arrives."),
    (
        "choice",
        "unclear",
        "After the update the app crashed while I was paying, and now I see two pending charges and I can't log back in.",
    ),
    (
        "choice",
        "unclear",
        "We'd like more seats, but the seat-management page won't load and our last invoice looks wrong.",
    ),
    ("choice", "unclear", "Can someone help me sort out my account? Things aren't right lately."),
    (
        "score",
        "clear",
        "Production checkout is down for every customer since 09:00; every order returns a 500 error.",
    ),
    ("score", "clear", "Typo on the settings page: 'Setings' instead of 'Settings'."),
    (
        "score",
        "clear",
        "Exports fail for our whole team and our board report is due tomorrow; no workaround found.",
    ),
    (
        "score",
        "unclear",
        "CSV export sometimes drops the last row on big files. Exporting twice seems to fix it. Not sure how many users are affected.",
    ),
    ("score", "unclear", "The dashboard is slow for me today. Might be my connection."),
    (
        "score",
        "unclear",
        "A colleague says notifications aren't arriving, but I haven't seen it myself.",
    ),
]
QUESTION = {
    "boolean": {"refund": Q_REFUND},
    "choice": {"route": Q_ROUTE},
    "score": {"severity": Q_SEV},
}

rows = []
for i, (qtype, clarity, text) in enumerate(cases):
    res = call(f"s3-{qtype}-{clarity}-{i}", text, QUESTION[qtype])
    ((name, ans),) = res.answers.items()
    if qtype == "boolean":
        p = ans["probability"]
        rows.append(
            {
                "question type": qtype,
                "clarity": clarity,
                "answer": f"P(yes)={p:.2f}",
                "highest probability": max(p, 1 - p),
                "confidence": None,
                "text": text[:60],
            }
        )
    else:
        probs = ans["probabilities"]
        pick = ans["choice"] if qtype == "choice" else f"score={ans['score']:.2f}"
        rows.append(
            {
                "question type": qtype,
                "clarity": clarity,
                "answer": pick,
                "highest probability": max(probs.values()),
                "confidence": ans["confidence"],
                "text": text[:60],
            }
        )
conf = pd.DataFrame(rows)
conf

,primitive,input,answer,top prob,confidence,text
0,boolean,clear,P(yes)=0.99,0.99,NaN,"I was charged twice, please refund the duplicate $49 payment"
1,boolean,clear,P(yes)=0.01,0.99,NaN,Love the new dashboard! Quick question: how do I change the
2,boolean,clear,P(yes)=0.98,0.98,NaN,"Cancel my plan and give me my money back, this product never"
3,boolean,ambiguous,P(yes)=0.25,0.75,NaN,"Hi, there's a $49 charge on my card from you. I'm not sure i"
4,boolean,ambiguous,P(yes)=0.06,0.94,NaN,The invoice looks higher than I expected. I'm not asking for
5,boolean,ambiguous,P(yes)=0.15,0.85,NaN,If the export bug isn't fixed by Friday I'll have to reconsi
6,choice,clear,billing,1.00,1.00,"I was charged twice for order #48213, please reverse one."
7,choice,clear,technical,1.00,1.00,The Analytics page throws a 500 error every time I open it.
8,choice,clear,account,1.00,1.00,I forgot my password and the reset email never arrives.
9,choice,ambiguous,technical,0.50,0.38,"After the update the app crashed while I was paying, and now"


In [8]:
conf.groupby(["question type", "clarity"])[["highest probability", "confidence"]].mean().round(2)

top prob  confidence
primitive input                          
boolean   ambiguous      0.85         NaN
          clear          0.99         NaN
choice    ambiguous      0.77        0.71
          clear          1.00        1.00
score     ambiguous      0.92        0.88
          clear          1.00        1.00

**What the tables show.** On clear tickets every question type is near certain (probabilities and
`confidence` at or close to 1). On unclear tickets the averages drop, but not evenly:

- **Choice:** the clearest drop is the ticket in the cell below (the app crashed during payment,
  there are two pending charges, and the customer cannot log in). The probability is split between
  `billing` and `technical`, and `confidence` is well under 0.5. The other two unclear choice
  tickets still got a confident single answer (about 0.85 to 0.9).
- **Yes/no:** the unclear refund messages did not move the probability towards 0.5. Jev read them
  as not asking for a refund (probability of yes between about 0.06 and 0.25). Yes/no answers have
  no separate `confidence` field, so distance from 0.5 is the only measure, and here it stayed wide.
- **Score:** the unclear bug reports scored near the middle of the scale, with `confidence` around
  0.85 to 0.95. That is a small fall from 1.0, not a large one.

I chose the tickets and decided which ones were unclear, so treat this as an illustration and not a
measurement. Jev's answers also vary slightly from run to run, by a few hundredths.

In [9]:
# One unclear choice input in full: the spread across options.
amb = next(c for c in cases if c[0] == "choice" and c[1] == "unclear")
print(amb[2])
res = call("s3-choice-unclear-detail", amb[2], {"route": Q_ROUTE})
show(res, "AMBIGUOUS CHOICE")

After the update the app crashed while I was paying, and now I see two pending charges and I can't log back in.


AMBIGUOUS CHOICE  latency=0.39s  tokens in/out=412/52  market cost=$0.0000173
{
  "route": {
    "type": "choice",
    "choice": "billing",
    "probabilities": {
      "billing": 0.52,
      "other": 0.03,
      "account": 0.04,
      "sales": 0,
      "technical": 0.41
    },
    "confidence": 0.38
  }
}


## 4. Several questions in one call versus separate calls

The same ticket thread and the same five questions, sent two ways: all five in one request, or five
separate requests. Each way is run three times, and we record the time taken and the input tokens.
The point is cost: separate calls send the ticket text again every time.

In [10]:
THREAD = (
    "Ticket #77120 - conversation so far\n\n"
    "[Customer, Mon 09:14] Hello, I upgraded from the Starter to the Team plan on the 3rd. Since "
    "then my dashboard has been loading extremely slowly, sometimes taking a minute, and twice "
    "this week it timed out entirely while I was presenting to my leadership team. That was "
    "embarrassing.\n"
    "[Agent, Mon 10:02] Sorry to hear that. Could you tell us which browser you use and whether "
    "the problem happens on every dashboard?\n"
    "[Customer, Mon 10:40] Chrome, latest version. It's every dashboard but worst on the sales "
    "one, which has about 40 charts. My colleague on the Starter plan doesn't see this at all, "
    "which makes me think the upgrade did something.\n"
    "[Agent, Mon 11:15] Thanks. We're looking at whether Team-plan workspaces are being routed "
    "through a different cluster. In the meantime you could try splitting the sales dashboard.\n"
    "[Customer, Tue 08:30] That's not a real fix. I'm paying $99 a month more for a worse "
    "product. Also, I noticed I was billed for the Team plan AND the Starter plan on the 3rd. If "
    "this isn't sorted by Thursday's board meeting, I want the difference refunded and I'll be "
    "looking at other vendors. Please escalate this to someone senior."
)
FIVE = {
    "refund": jev.boolean("Is the customer asking for a refund?"),
    "frustrated": jev.boolean("Is the customer frustrated or angry?"),
    "escalate": jev.boolean("Has the customer explicitly asked for escalation to someone senior?"),
    "route": jev.choice("Which team should handle this ticket first?", DEPARTMENTS),
    "urgency": jev.score(
        "How urgent is this ticket?",
        [
            "low: no deadline, nobody blocked",
            "medium: degraded service or a deadline within a week",
            "high: customer is blocked now or threatens to leave imminently",
        ],
    ),
}
REPS = 3

one_call_rows, separate_rows = [], []
one_call_answers, separate_answers = None, {}
for rep in range(REPS):
    res = call(f"s4-one-call-rep{rep}", THREAD, FIVE)
    one_call_rows.append((res.latency_s, res.input_tokens, res.output_tokens, res.market_cost))
    one_call_answers = res.answers

    lat = inp = out = cost = 0
    for name, q in FIVE.items():
        r = call(f"s4-separate-{name}-rep{rep}", THREAD, {name: q})
        lat += r.latency_s
        inp += r.input_tokens
        out += r.output_tokens
        cost += r.market_cost
        separate_answers.update(r.answers)
    separate_rows.append((lat, inp, out, cost))


def summarise(rows):
    return {
        "latency_s (median)": round(statistics.median(r[0] for r in rows), 2),
        "input tokens (median)": statistics.median(r[1] for r in rows),
        "output tokens (median)": statistics.median(r[2] for r in rows),
        "market cost $ (median)": round(statistics.median(r[3] for r in rows), 7),
    }


comparison = pd.DataFrame(
    {
        "1 call x 5 questions": summarise(one_call_rows),
        "5 calls x 1 question": summarise(separate_rows),
    }
)
comparison["ratio (5 calls / 1 call)"] = (comparison.iloc[:, 1] / comparison.iloc[:, 0]).round(2)
comparison

,1 call x 5 questions,5 calls x 1 question,ratio (5 calls / 1 call)
latency_s (median),0.460000,2.060000,4.48
input tokens (median),795.000000,3015.000000,3.79
output tokens (median),119.000000,134.000000,1.13
market cost $ (median),0.000033,0.000127,3.79


In [11]:
# Do the two ways agree on the answers? (Last repetition of each.)
def main_answer(a):
    if a["type"] == "boolean":
        return round(a["probability"], 2)
    if a["type"] == "choice":
        return a["choice"]
    return round(a["score"], 2)


pd.DataFrame(
    {
        "one call": {k: main_answer(v) for k, v in one_call_answers.items()},
        "separate calls": {k: main_answer(v) for k, v in separate_answers.items()},
    }
)

,fan-out,sequential
refund,0.95,0.94
frustrated,0.98,0.98
escalate,0.99,0.99
route,billing,billing
urgency,1.91,1.9


In [12]:
print("per run (latency_s, in_tokens, out_tokens, $)")
print("one call  :", [tuple(round(x, 4) for x in r) for r in one_call_rows])
print("separate calls:", [tuple(round(x, 4) for x in r) for r in separate_rows])

per-rep (latency_s, in_tokens, out_tokens, $)
fan-out   : [(0.3928, 795, 119, 0.0), (0.462, 795, 119, 0.0), (0.5172, 795, 119, 0.0)]
sequential: [(2.1222, 3015, 134, 0.0001), (2.0606, 3015, 134, 0.0001), (1.9595, 3015, 134, 0.0001)]


**Result.** In this run, sending all five questions together took about 0.46 seconds (median),
against about 2.06 seconds for five separate calls, and used 795 input tokens against 3,015. The
token saving is smaller than five times because each question still adds its own text. The answers
were the same to within about 0.01. Times are for the successful attempt only, so retries after 429
errors are not counted.

## 5. Where it does not work well

Following are tasks that (according to docs) Jev is not well suited to. Results weren't generally too bad though...

### 5a. Counting

Server logs containing a known number of `timeout` lines. Jev is asked to pick a range for the
count, and the answer is checked against the true number. The logs are generated with a fixed
random seed.

In [13]:
import random

rng = random.Random(7)
FILLER = [
    "INFO  request served path=/api/orders status=200",
    "INFO  cache refresh complete entries=1204",
    "WARN  slow query took 812ms table=invoices",
    "INFO  user login ok user_id=42",
    "DEBUG heartbeat ok node=worker-3",
]


def make_log(n_timeouts, n_lines=40):
    lines = [rng.choice(FILLER) for _ in range(n_lines - n_timeouts)]
    for _ in range(n_timeouts):
        lines.insert(rng.randint(0, len(lines)), "ERROR upstream call failed: timeout after 30s")
    return "\n".join(lines)


RANGES = {
    "1-3": "one to three lines",
    "4-6": "four to six lines",
    "7-9": "seven to nine lines",
    "10-12": "ten to twelve lines",
}


def count_range(n):
    return next(k for k in RANGES if int(k.split("-")[0]) <= n <= int(k.split("-")[1]))


rows = []
for n in [2, 3, 4, 5, 6, 7, 8, 9, 10, 12]:
    log = make_log(n)
    assert log.count("timeout") == n
    res = call(
        f"s5-count-{n}",
        log,
        {"count": jev.choice("How many log lines contain the word 'timeout'?", RANGES)},
    )
    a = res.answers["count"]
    rows.append(
        {
            "true count": n,
            "true range": count_range(n),
            "predicted": a["choice"],
            "correct": a["choice"] == count_range(n),
            "highest probability": max(a["probabilities"].values()),
            "confidence": a["confidence"],
        }
    )
counting = pd.DataFrame(rows)
counting

,true count,true bucket,predicted,correct,top prob,confidence
0,2,1-3,1-3,True,1.00,1.00
1,3,1-3,1-3,True,0.81,0.74
2,4,4-6,4-6,True,0.99,0.99
3,5,4-6,4-6,True,0.99,0.99
4,6,4-6,7-9,False,0.60,0.46
5,7,7-9,7-9,True,0.77,0.69
6,8,7-9,7-9,True,0.77,0.69
7,9,7-9,7-9,True,0.56,0.42
8,10,10-12,10-12,True,0.68,0.56
9,12,10-12,10-12,True,0.94,0.92


In [14]:
print(f"accuracy: {counting.correct.mean():.0%}")
print(counting.groupby("correct")[["highest probability", "confidence"]].mean().round(2))

accuracy: 90%
         top prob  confidence
correct                      
False        0.60        0.46
True         0.83        0.78


Choosing a range is forgiving: a count of 6 only has to land in "4-6". A harder version uses longer
logs (80 lines) and asks for the exact count, chosen from 1 to 15.

In [15]:
EXACT = {str(k): f"exactly {k} lines" for k in range(1, 16)}
rows = []
for n in [3, 5, 7, 8, 9, 11, 13, 14]:
    log = make_log(n, n_lines=80)
    assert log.count("timeout") == n
    res = call(
        f"s5-count-exact-{n}",
        log,
        {"count": jev.choice("How many log lines contain the word 'timeout'?", EXACT)},
    )
    a = res.answers["count"]
    rows.append(
        {
            "true count": n,
            "predicted": int(a["choice"]),
            "correct": int(a["choice"]) == n,
            "error": abs(int(a["choice"]) - n),
            "highest probability": max(a["probabilities"].values()),
            "confidence": a["confidence"],
        }
    )
counting_exact = pd.DataFrame(rows)
counting_exact

,true count,predicted,correct,abs error,top prob,confidence
0,3,3,True,0,0.94,0.93
1,5,5,True,0,0.53,0.48
2,7,6,False,1,0.53,0.48
3,8,8,True,0,0.36,0.30
4,9,10,False,1,0.22,0.15
5,11,11,True,0,0.21,0.16
6,13,11,False,2,0.21,0.14
7,14,11,False,3,0.23,0.18


In [16]:
print(f"exact-count accuracy: {counting_exact.correct.mean():.0%}")
print(counting_exact.groupby("correct")[["highest probability", "confidence"]].mean().round(2))

exact-count accuracy: 50%
         top prob  confidence
correct                      
False        0.30        0.24
True         0.51        0.47


**Counting.** When choosing a range on 40-line logs, Jev was right most of the time (every time in
an earlier run of this notebook, all but one in this run). Its confidence was lower near the edges
of the ranges and on the miss. Exact counts on 80-line logs go badly: about half are right, the
misses are off by up to 3, and `confidence` is low throughout, mostly below 0.5. So here Jev is not
confidently wrong, and the confidence value gives a warning. It is also cautious, since several
correct answers had similarly low confidence.

### 5b. Single-word levels versus concrete descriptions in a score question

What happens when the descriptions of scale are under-specified? Here we look at the same bug reports, with two versions of the scale. One uses single-word levels ("mild", "moderately severe", "severe"). The other describes each level with a concrete situation, as in the rest of this notebook. The "intended" level for each report is my own judgement, based on the concrete descriptions: 0 = low, 1 = medium, 2 = high.

In [21]:
SINGLE_WORD = ["mild", "moderately severe", "severe"]
reports = [
    ("Footer link colour is slightly off-brand on the pricing page.", 0),
    ("Typo in a tooltip: 'Recieve notifications'.", 0),
    ("CSV export drops the last row for files over 5k rows; exporting twice works around it.", 1),
    ("Dashboard charts take 40s to load for the Team plan; a demo is due on Thursday.", 1),
    ("Checkout returns a 500 for every customer since 09:00. No orders are going through.", 2),
    ("Any logged-in user can view other customers' invoices by editing the URL id.", 2),
]
rows = []
for i, (text, intended) in enumerate(reports):
    for name, crit in (("single-word", SINGLE_WORD), ("concrete", SEVERITY)):
        res = call(
            f"s5-score-{name}-{i}", text, {"sev": jev.score("Rate the severity of this bug.", crit)}
        )
        a = res.answers["sev"]
        rows.append(
            {
                "report": text[:55],
                "intended": intended,
                "wording": name,
                "score": a["score"],
                "error": abs(a["score"] - intended),
                "confidence": a["confidence"],
            }
        )
scores = pd.DataFrame(rows)
scores.pivot(index="report", columns="wording", values=["score", "confidence"]).round(2)

score          confidence         
scale                                                    bare concrete       bare concrete
report                                                                                    
Any logged-in user can view other customers' invoices b  1.97     2.00       0.95     0.99
CSV export drops the last row for files over 5k rows; e  1.26     1.01       0.58     0.95
Checkout returns a 500 for every customer since 09:00.   2.00     2.00       1.00     1.00
Dashboard charts take 40s to load for the Team plan; a   1.52     1.00       0.28     0.99
Footer link colour is slightly off-brand on the pricing  0.00     0.00       1.00     1.00
Typo in a tooltip: 'Recieve notifications'.              0.00     0.00       1.00     1.00

In [22]:
scores.groupby("wording")[["error", "confidence"]].mean().round(2)

,abs error,confidence
scale,,
bare,0.14,0.80
concrete,0.00,0.99


**Single-word levels versus concrete descriptions.** For the extreme reports (a cosmetic typo, an outage, a security hole) both scales give the same score. The difference is in the middle. With single-word levels, the two medium-severity reports score higher (towards "moderately severe" and "severe") and Jev is much less confident than with concrete descriptions (roughly 0.3 to 0.6 in this run, against about 0.95 to 1.0). The intended levels are my own, based on the concrete descriptions, so the error column favours the concrete scale by construction. The difference in confidence is the more useful result. There are only a few reports, so this is an illustration.

**Overall for section 5.** The problems are real but in these tests they showed up as low confidence, not as confident wrong answers. The samples are small and informal. They do not show that confidence is well calibrated, they show only that confidence did not stay high where accuracy fell.

## API failures during this run

Temporary failures (HTTP 429 and 5xx) were retried, waiting longer after each failure. Every
attempt, including failed ones, is in `results/phase1_raw.jsonl`.

In [23]:
n_calls = sum(1 for _ in RAW_LOG.open())
print(f"total HTTP attempts: {n_calls}; failed attempts: {len(FAILURES)}")
for label, attempt, status in FAILURES:
    print(f"  {label}: attempt {attempt} -> HTTP {status}")

total HTTP attempts: 137; failed attempts: 50
  s4-fanout-rep0: attempt 1 -> HTTP 503
  s4-sequential-frustrated-rep0: attempt 1 -> HTTP 429
  s4-sequential-escalate-rep0: attempt 1 -> HTTP 429
  s4-sequential-escalate-rep0: attempt 2 -> HTTP 429
  s4-sequential-route-rep0: attempt 1 -> HTTP 429
  s4-sequential-route-rep0: attempt 2 -> HTTP 429
  s4-sequential-route-rep0: attempt 3 -> HTTP 429
  s4-sequential-urgency-rep0: attempt 1 -> HTTP 429
  s4-fanout-rep1: attempt 1 -> HTTP 429
  s4-fanout-rep1: attempt 2 -> HTTP 429
  s4-fanout-rep1: attempt 3 -> HTTP 429
  s4-sequential-refund-rep1: attempt 1 -> HTTP 429
  s4-sequential-escalate-rep1: attempt 1 -> HTTP 429
  s4-fanout-rep2: attempt 1 -> HTTP 429
  s4-sequential-refund-rep2: attempt 1 -> HTTP 429
  s4-sequential-refund-rep2: attempt 2 -> HTTP 429
  s4-sequential-refund-rep2: attempt 3 -> HTTP 429
  s4-sequential-frustrated-rep2: attempt 1 -> HTTP 429
  s4-sequential-frustrated-rep2: attempt 2 -> HTTP 429
  s4-sequential-urgency-

About a third of HTTP attempts failed with temporary errors, mostly 429 ("upstream provider is
currently experiencing high demand") and one 503. The run was made five days after Jev's release.
Every call succeeded within the retry limit, so no result above is missing or replaced. The times
shown in this notebook are for the successful attempt only and exclude retries and waiting.

**For Phase 2:** 1,000 rows across three datasets and two conditions will need proper pacing, a retry
policy, and failure counts reported for each condition.